<a href="https://colab.research.google.com/github/cbonnin88/ThreadFlip/blob/main/Product_Metrics_ThreadFlip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from statsmodels.stats.proportion import proportions_chisquare

# **Generating Experimentaiton Simulation Data**

In [ ]:
np.random.seed(42)
n_samples = 5000

In [ ]:
product_data = pl.DataFrame({
    'user_id': np.arange(10000,10000 + n_samples),
    'test_group': ['Control'] * (n_samples // 2) + ['Variant'] * (n_samples // 2),
    'converted': np.append(
        np.random.choice([0,1], size=n_samples // 2, p=[0.88,0.12]), # 112% baseline conversion
        np.random.choice([0,1], size=n_samples // 2, p=[0.85,0.15]) # 15% variant conversion
    ),
    'order_value': np.append(
        np.random.normal(loc=42.50,scale=10,size=n_samples // 2),
        np.random.normal(loc=48.50,scale=12, size=n_samples // 2)
    ),
    'csat_score': np.append(
        np.random.choice(np.arange(1,11), size=n_samples // 2, p=[0.02,0.03,0.05,0.05,0.1,0.15,0.2,0.25,0.1,0.05]),
        np.random.choice(np.arange(1, 11), size=n_samples // 2, p=[0.01,0.02,0.02,0.05,0.05,0.1,0.2,0.3,0.15,0.1])
    )
})

In [ ]:
display(product_data.head())

user_id,test_group,converted,order_value,csat_score
i64,str,i64,f64,i64
10000,"""Control""",0,36.530257,5
10001,"""Control""",1,18.596956,2
10002,"""Control""",0,38.377793,9
10003,"""Control""",0,51.634737,6
10004,"""Control""",0,47.876299,7


In [ ]:
print(product_data.shape)

(5000, 5)


**Data Aggregations (Polars)**

In [ ]:
metrics_summary = product_data.group_by('test_group').agg([
    pl.len().alias('sample_size'),
    pl.col('converted').sum().alias('total_conversions'),
    pl.col('converted').mean().alias('conversion_rate'),
    pl.col('order_value').mean().round(2).alias('average_order_value'),
    pl.col('csat_score').mean().alias('mean_cst')
])

In [ ]:
print('--- A/B Testing Cohort Summary Statistics ---')
display(metrics_summary)

--- A/B Testing Cohort Summary Statistics ---


test_group,sample_size,total_conversions,conversion_rate,average_order_value,mean_cst
str,u32,i64,f64,f64,f64
"""Variant""",2500,345,0.138,48.51,7.3044
"""Control""",2500,294,0.1176,42.12,6.6128


# **Quantitiative Statistical Significance Hypopthesis Testing**


Hypothesis:
- H0: Smart Bundle Feature does not change conversion rates.
- H1: Smart Bundle Feature significantly improves conversion rates.

In [ ]:
control_cohort = metrics_summary.filter(pl.col('test_group') == 'Control')
variant_cohort = metrics_summary.filter(pl.col('test_group') == 'Variant')

In [ ]:
count = np.array([control_cohort['total_conversions'][0], variant_cohort['total_conversions'][0]])
nobs = np.array([control_cohort['sample_size'][0], variant_cohort['sample_size'][0]])

In [ ]:
stat, p_value, _ = proportions_chisquare(count, nobs)
alpha = 0.05

In [ ]:
print('\n--- Experimentation Statistical Significance Testing Outcomes ---')
print(f'Calculated Chi-Square Statistic: {stat:.4f}')
print(f'Experiment p-value: {p_value:.4f}')


--- Experimentation Statistical Significance Testing Outcomes ---
Calculated Chi-Square Statistic: 4.6668
Experiment p-value: 0.0308


In [ ]:
if p_value < alpha:
  print('Result: Reject H0. The Variant shows a statistically significant improvement in checkout conversion !')
else:
  print('Results: Fail to reject H0. NO statistically significant conversion variance identified.')

Result: Reject H0. The Variant shows a statistically significant improvement in checkout conversion !


# **Data Visualization**

In [ ]:
fig = px.bar(
    product_data.to_pandas(),
    x='test_group',
    y='order_value',
    color='test_group',
    title='Impact of Smart Bundle UI Deployment on Customer Average Order Value (€)',
    color_discrete_sequence = ["#7F8C8D", "#2ECC71"]
)

fig.show()